# UK road collision severity prediction

This notebook is a lightweight, executable entry point to the reproducible project.
It reads the tracked result snapshot and report figures instead of duplicating the
training implementation. The workflow has four stages: source-data validation,
descriptive analysis, five-model temporal validation, and held-out evaluation with
additional diagnostics.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

START = Path.cwd().resolve()
ROOT = next(
    (path for path in [START, *START.parents] if (path / "configs/default.yaml").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project repository.")

snapshot = json.loads(
    (ROOT / "reports/results_snapshot.json").read_text(encoding="utf-8")
)
print(f"Repository: {ROOT.name}")
print(f"Snapshot generated: {snapshot['snapshot_generated_at_utc']}")

Repository: uk-road-collision-severity-prediction
Snapshot generated: 2026-08-31T15:14:02.327952+00:00


## 1. Data provenance

The source checksum, processing contract and row count below come from the tracked
result snapshot. Regenerate the snapshot after rebuilding the data or models.

In [2]:
data = snapshot["data"]
display(pd.DataFrame(
    {
        "Value": [
            data["source_file"],
            data["source_sha256"],
            data["rows"],
            " to ".join(data["date_range"]),
            f"{data['ksi_share']:.2%}",
            data["contract_version"],
        ]
    },
    index=[
        "Source file",
        "Source SHA-256",
        "Processed rows",
        "Date range",
        "KSI share",
        "Data contract",
    ],
))

,Value
Source file,data/raw/dft-road-casualty-statistics-collisio...
Source SHA-256,8ce3f1290ea4830c041ddd737b543fb06b8667215208a1...
Processed rows,513801
Date range,2021-01-01 to 2025-12-31
KSI share,24.21%
Data contract,dft_open_dataset_2025


## 2. Descriptive analysis

The tracked figures remain visible when the notebook is viewed without execution.

![Severity composition](../reports/figures/processed/01_severity_composition.png)

![Monthly collision volume and KSI trend](../reports/figures/processed/20_monthly_time_series.png)

![Lighting conditions](../reports/figures/processed/06_ksi_by_light.png)

![Spatial collision density and severity](../reports/figures/processed/19_spatial_hex_analysis.png)

## 3. Five-model temporal validation

All candidates use 2021–2023 for training and 2024 for model comparison. Average
Precision is the primary selection metric; 2025 remains untouched during selection.

In [3]:
comparison = pd.DataFrame(snapshot["validation"]["models"])
comparison["model"] = comparison["model"].str.replace("_", " ").str.title()
comparison = comparison.rename(columns={
    "rank": "Rank",
    "model": "Model",
    "roc_auc": "ROC-AUC",
    "average_precision": "Average precision",
    "brier_score": "Brier score",
    "training_seconds": "Training seconds",
})
display(comparison.round(4))

,Rank,Model,ROC-AUC,Average precision,Brier score,Training seconds
0,1,Lightgbm,0.6558,0.3798,0.2311,5.9374
1,2,Catboost,0.6557,0.3792,0.2296,90.0490
2,3,Extra Trees,0.6443,0.3670,0.2242,28.5999
3,4,Logistic Regression,0.5894,0.3244,0.2431,7.0069
4,5,Dummy,0.5000,0.2484,0.1869,2.5112


![Validation model comparison](../reports/figures/model/model_validation_comparison.png)

![Chronological learning curve](../reports/figures/model/temporal_learning_curve.png)

## 4. Held-out 2025 evaluation

The classification threshold is selected on 2024 by maximising KSI F1, then applied
once to the 2025 test year. Probabilities require calibration before literal risk
interpretation.

In [4]:
test = snapshot["test"]
metrics = [
    ("Selected model", test["selected_model"]),
    ("Threshold", test["threshold"]),
    ("ROC-AUC", test["roc_auc"]),
    ("Average precision", test["average_precision"]),
    ("Brier score", test["brier_score"]),
    ("Balanced accuracy", test["balanced_accuracy"]),
    ("KSI precision", test["ksi_precision"]),
    ("KSI recall", test["ksi_recall"]),
    ("KSI F1", test["ksi_f1"]),
]
display(pd.DataFrame(metrics, columns=["Metric", "Result"]).round(4))

,Metric,Result
0,Selected model,lightgbm
1,Threshold,0.46
2,ROC-AUC,0.639312
3,Average precision,0.381905
4,Brier score,0.234938
5,Balanced accuracy,0.594959
6,KSI precision,0.328042
7,KSI recall,0.70042
8,KSI F1,0.446817


![Permutation importance](../reports/figures/model/permutation_importance.png)

![Precision-recall curve](../reports/figures/model/precision_recall_curve.png)

![Confusion matrix](../reports/figures/model/confusion_matrix.png)

![Calibration curve](../reports/figures/model/calibration_curve.png)

## Reproduce the workflow

Run these commands from the repository root:

    python scripts/download_data.py
    python scripts/01_raw_analysis_and_processing.py
    python scripts/02_processed_analysis_and_visualisation.py
    python scripts/tune_lightgbm.py
    python scripts/03_model_training_and_visualisation.py
    python scripts/04_additional_visual_analysis.py
    python scripts/build_results_snapshot.py
    python -m pytest -q

See [README.md](../README.md) for setup, interpretation boundaries and output paths,
and [the full figure narrative](../reports/figure_story.md) for detailed findings.